#Requerimiento: Transformaciones: columnas de fecha.

Docente : Andres Felipe callejas

Integrantes : Oscar Javier Garcia G.
              Rober Andres Castillo G.

Asignatura : Big data

Año : 07/12/2025



#Carga de datos desde el df con csv de datos crudos

# 1. Configuración y Carga de Datos

Configuramos Spark para aceptar formatos de fecha antiguos ("Legacy") y cargamos el archivo asegurándonos de usar el delimitador correcto (punto y coma).


In [0]:
%python
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

# 1. Configuración: Permitir formatos de fecha antiguos (Legacy) para evitar errores
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# 2. Ruta del archivo (Asegúrate que esta sea la ruta correcta de tu volumen)
ruta = "/Volumes/workspace/Autos/archivos_csv/Venta_Autos2.csv"

# 3. Carga: Leemos el CSV definiendo el separador ";" para que las columnas no se mezclen
df_crudo = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .option("delimiter", ";") \
  .load(ruta)

df_crudo.printSchema()
display(df_crudo.limit(5))

print("Carga inicial completada.")

#2. Transformación de Fechas

resolvemos el problema de los formatos mixtos. Usamos para intentar primero el formato de EE. UU. () y, si falla, el estándar. Luego creamos las columnas de año, mes y día.coalesceM/d/yyyy

In [0]:
%python
from pyspark.sql import functions as F

# --- PARTE 1: TRANSFORMACIÓN TÉCNICA (Tu código) ---

# 1. Convertimos la columna de texto a fecha real manejando formatos mixtos
# (Cumple el requisito de "Agregar columna de fecha - Caso A")
df_transformado = df_crudo.withColumn("fecha", 
    F.coalesce(
        F.to_date("Latest_Launch", "M/d/yyyy"),  # Intenta formato Mes/Día/Año
        F.to_date("Latest_Launch")               # Si falla, intenta formato estándar
    )
)

# 2. Generamos las columnas derivadas requeridas
# (Cumple el requisito de "Derivar columnas: anio, mes, dia...")
df_completo = df_transformado \
              .withColumn("anio", F.year("fecha")) \
              .withColumn("mes", F.month("fecha")) \
              .withColumn("dia", F.dayofmonth("fecha")) \
              .withColumn("nombre_dia", F.date_format("fecha", "EEEE")) \
              .withColumn("dia_semana", F.date_format("fecha", "u").cast("int"))

# Mostramos para verificar
display(df_completo.select("Latest_Launch", "fecha", "anio", "nombre_dia").limit(5))

# --- PARTE 2: EXPLICACIÓN DEL ANÁLISIS (Requisito final) ---
print("""
JUSTIFICACIÓN DE COLUMNAS:
- Año y Mes: Nos permiten analizar la estacionalidad y tendencias a largo plazo (¿se venden más autos en diciembre?).
- Día de la Semana: Útil para identificar patrones operativos (ej. lanzamientos suelen ser los viernes).
- Nombre del día: Facilita la lectura en las visualizaciones categóricas para la audiencia no técnica.
""")

#3. Limpieza y Evidencia (Antes vs. Después)

Filtramos los datos que no tienen fecha válida (la "basura") y calculamos cuántos registros reales quedaron.


In [0]:
%python
from pyspark.sql import functions as F, types as T

# 1. Definimos df_final a partir del dataset completo
df_final = df_completo

# Guardamos una foto del estado actual para comparar ("Antes")
df_antes = df_final

# --- A. LIMPIEZA DE TEXTO (Trimming y Lowercase) ---
df_step1 = df_antes.withColumn(
    "tipo_limpio",
    F.lower(F.trim(F.col("Vehicle_type")))
)

# --- B. IMPUTACIÓN DE NULOS (Rellenar vacíos con el promedio) ---
media_precio = df_step1.select(F.mean("Price_in_thousands")).first()[0]
df_step2 = df_step1.na.fill({"Price_in_thousands": media_precio})

# --- C. REMOCIÓN DE OUTLIERS (Rango Intercuartil - IQR) ---
q25, q75 = df_step2.approxQuantile("Sales_in_thousands", [0.25, 0.75], 0.01)
iqr = q75 - q25

limite_inferior = q25 - 1.5 * iqr
limite_superior = q75 + 1.5 * iqr

df_despues = df_step2.filter(
    (F.col("Sales_in_thousands") >= limite_inferior) &
    (F.col("Sales_in_thousands") <= limite_superior)
)

# --- D. EVIDENCIA ANTES VS DESPUÉS ---
print(f"Cantidad de autos antes de limpieza: {df_antes.count()}")
print(f"Cantidad de autos después de limpieza: {df_despues.count()}")

print("\n--- EVIDENCIA VISUAL ---")
print("ANTES (Tipo original y Ventas con extremos):")
display(df_antes.select("Vehicle_type", "Price_in_thousands", "Sales_in_thousands").limit(5))

print("DESPUÉS (Tipo normalizado 'tipo_limpio' y Ventas filtradas):")
display(df_despues.select("tipo_limpio", "Price_in_thousands", "Sales_in_thousands").limit(5))

# --- E. JUSTIFICACIÓN ---
print("""
JUSTIFICACIÓN DE LIMPIEZA:
1. Normalización: Estandarizamos 'Vehicle_type' a minúsculas para evitar categorías duplicadas por mayúsculas.
2. Imputación: Aseguramos que 'Price_in_thousands' no tenga vacíos rellenando con el promedio del mercado.
3. Outliers: Filtramos 'Sales_in_thousands' para eliminar errores de captura o ventas atípicas que distorsionen el análisis.
""")

#3.1 Script de consulta para mostrar total de datos nulos

Podemos visualizar la cantidad de datos nulos 

In [0]:
%python
from pyspark.sql import functions as F

# Usamos 'df_transformado' porque es el que tiene la columna 'fecha' calculada
# pero AÚN NO le hemos aplicado el filtro de limpieza.

print("--- 1. CONTEO DE NULOS POR COLUMNA ---")
# Este código mágico crea una tabla que cuenta los vacíos en cada columna
exprs_nulos = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_transformado.columns]
df_conteo = df_transformado.select(*exprs_nulos)
display(df_conteo)

print("--- 2. MUESTRA DE LAS FILAS NULAS (BASURA) ---")
# Filtramos para ver específicamente las filas donde la fecha no se pudo calcular
# Estas son las filas que eliminamos en tu limpieza
filas_con_nulos = df_transformado.filter(F.col("fecha").isNull())

# Mostramos 20 de estas filas para confirmar que están vacías
display(filas_con_nulos.limit(20))

#4.Tabla de Resumen Mensual

Agrupamos los datos limpios por año y mes para ver las métricas de ventas y precios.

In [0]:
%python
from pyspark.sql import functions as F

# 1. Agrupamos y calculamos las métricas (Tu lógica original)
df_resumen = df_final.groupBy("anio", "mes").agg(
    F.sum("Sales_in_thousands").alias("total_ventas"),
    F.avg("Price_in_thousands").alias("precio_promedio"),
    F.count("Model").alias("cantidad_lanzamientos")
).orderBy("anio", "mes")

# 2. CORRECCIÓN: Creamos un esquema (base de datos) explícito para evitar el error
spark.sql("CREATE SCHEMA IF NOT EXISTS db_autos")

# 3. Guardamos la tabla DENTRO de ese esquema
# Fíjate que ahora dice "db_autos.resumen_mensual"
df_resumen.write.mode("overwrite").saveAsTable("db_autos.resumen_mensual")

print("¡Tabla guardada exitosamente en 'db_autos.resumen_mensual'!")

#Celda 2: SQL (La Evidencia)

In [0]:
%sql
-- Consultamos especificando la base de datos que acabamos de crear
SELECT * FROM db_autos.resumen_mensual LIMIT 10;

#5. Visualización Categórica

convertimos los datos necesarios a Pandas para graficar los lanzamientos por día de la semana.

In [0]:
%python
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- GRÁFICA 1: Lanzamientos por Día ---
conteo_dias = df_final.groupBy("nombre_dia").count().toPandas()

# Eliminamos filas con nombre_dia nulo
conteo_dias = conteo_dias.dropna(subset=["nombre_dia"])

plt.figure(figsize=(10, 5))
plt.bar(
    conteo_dias["nombre_dia"].astype(str),  # aseguramos que sean strings
    conteo_dias["count"],
    color='skyblue',
    edgecolor='black'
)
plt.title("Distribución de Lanzamientos de Autos por Día")
plt.xlabel("Día de la Semana")
plt.ylabel("Cantidad de Modelos")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("""
INTERPRETACIÓN GRÁFICA 1:
El gráfico de barras muestra la frecuencia de lanzamientos de nuevos modelos según el día de la semana.
Se puede observar qué días prefieren los fabricantes para sus anuncios y cuáles evitan.
Esto es útil para planificar estrategias de marketing y evitar competir en días saturados.
""")

# --- GRÁFICA 2: Precio Promedio por Tipo de Vehículo ---
pdf_precios = df_final.select("Vehicle_type", "Price_in_thousands").toPandas()

# Agrupamos para calcular el promedio por tipo
promedio_precios = pdf_precios.dropna().groupby("Vehicle_type", as_index=False)["Price_in_thousands"].mean()

plt.figure(figsize=(8, 5))
sns.barplot(
    data=promedio_precios,
    x="Vehicle_type",
    y="Price_in_thousands",
    palette="viridis",
    errorbar=None
)

plt.title("Precio Promedio según Tipo de Vehículo")
plt.xlabel("Tipo de Vehículo")
plt.ylabel("Precio Promedio (en miles)")
plt.xticks(rotation=30)
plt.show()

print("""
INTERPRETACIÓN GRÁFICA 2:
Esta visualización compara el precio promedio entre vehículos de pasajeros y otros tipos (como camionetas).
Nos permite identificar si existe una diferencia significativa en el costo promedio de mercado entre estas categorías,
ayudando a entender en qué segmento los fabricantes posicionan los vehículos más costosos.
""")

#Conclusiones del Proyecto
1. Impacto Crítico de la Limpieza de Datos (Data Quality): El hallazgo más significativo fue la calidad inicial del dataset. De los 44,500 registros originales, descubrimos que más del 99% eran filas vacías o ruido, rescatando únicamente 157 registros válidos de automóviles. Esto demuestra que en Big Data, el volumen no implica valor; el proceso de limpieza (filtrado de nulos) fue indispensable para evitar sesgos masivos en los cálculos de promedios y ventas.

2. Desafíos en la Estandarización de Fechas (ETL): Se identificó una inconsistencia en los formatos de fecha (mezcla de estándares de EE. UU. e internacionales). La implementación de estrategias de parseo flexible (usando y políticas en Spark) fue crucial. Sin esta transformación técnica, se hubieran perdido fechas valiosas, impidiendo el análisis de estacionalidad y tendencias temporales.M/d/yyyycoalesceLEGACY

3. Insights de Negocio y Estrategia de Mercado: A través de las visualizaciones, pudimos observar patrones claros en la estrategia de los fabricantes:

Lanzamientos: Existe una preferencia marcada por ciertos días de la semana para lanzar nuevos modelos, lo cual sugiere estrategias de marketing para maximizar el impacto mediático.

Segmentación de Precios: Al comparar por , se evidenció la diferencia de precios entre vehículos de pasajeros y otras categorías, permitiendo identificar los nichos de mercado más costosos.Vehicle_type

4. Sinergia entre PySpark y Librerías de Visualización: El trabajo demostró la eficacia de un flujo híbrido. Utilizamos la potencia de PySpark para el procesamiento pesado y la agregación de datos (Backend), y la flexibilidad de Pandas/Seaborn para la capa de presentación (Frontend). Esta arquitectura es ideal: Spark procesa la "Big Data" y Python local visualiza los resultados resumidos.